### Sentence Transformer on synonsis

In [90]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
animes = pd.read_parquet('data/animes.parquet')

In [ ]:
animes.synopsis.isna().sum()

np.int64(637)

In [ ]:
animes['synopsis'] = animes['synopsis'].fillna('')
animes['synopsis_length'] = animes['synopsis'].apply(lambda x: len(x.split()))

In [5]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
# using all-mpnet-base-v2 as it allow max_sequence lenght to be 512
model = SentenceTransformer("all-mpnet-base-v2")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(animes.synopsis.to_list(), show_progress_bar=True)
print(embeddings.shape)

Batches:   0%|          | 0/467 [00:00<?, ?it/s]

(14920, 768)


In [ ]:
# Save
np.save('embeddings.npy', embeddings)

In [33]:
embeddings = np.load('embeddings.npy')

In [93]:

# Content based recommendation system
def get_cbr(titles, animes, embeddings, top_n=20, exclude_franchise=True):
    # get indices for all selected titles
    indices = animes[animes['title'].isin(titles)].index.tolist()
    
    # average their embeddings into one query vector
    query_embedding = embeddings[indices].mean(axis=0).reshape(1, -1)
    
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    similar_indices = similarities.argsort()[::-1]
    
    recs = animes.iloc[similar_indices]
    
    if exclude_franchise:
        base_titles = animes.iloc[indices]['base_title'].tolist()
        recs = recs[~recs['base_title'].isin(base_titles)]
    
    # exclude the selected anime themselves
    recs = recs[~recs['title'].isin(titles)]
    
    return recs.head(top_n)



In [89]:
def search_by_synopsis(query, animes, embeddings, model, top_n=5):
    # embed the user's query
    query_embedding = model.encode([query])
    
    # compute similarity against all anime embeddings
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    
    # get top N most similar
    top_indices = similarities.argsort()[::-1][:top_n]
    
    recs = animes.iloc[top_indices]
    recs['similarity'] = similarities[top_indices]
    
    return recs